# kafka_sentiment.API.ipynb
## API Reference: Tweet Sentiment Analysis using Kafka and HuggingFace
**Author**: Aashish Vinod  
**Course**: DATA605 Spring 2026

## Why I built this

As a grad student I have used Kafka before but never combined it with an NLP model.
The idea of classifying tweet sentiment in real time as messages flow through Kafka
is a genuinely useful pattern - companies like Twitter and Reddit use similar
architectures to monitor public opinion at scale.

The hardest part was getting the HuggingFace model to run inside Docker with the
correct torch version and enough memory allocated.

## Notebook structure
1. Dataset loading API
2. Tweet preprocessing API
3. HuggingFace sentiment model API
4. Kafka Producer API
5. Kafka Consumer API
6. Utility Functions API

## 1. Imports and Setup

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka_sentiment_utils import (
    load_sentiment140, preprocess_tweet, create_tweet_event,
    serialize_event, deserialize_event,
    compute_sentiment_stats, format_sentiment_summary,
    TOPIC_NAME, SENTIMENT_LABELS, SENTIMENT_COLORS,
)
print('All imports successful!')
print(f'Kafka topic: {TOPIC_NAME}')
print(f'Sentiment labels: {SENTIMENT_LABELS}')


All imports successful!
Kafka topic: tweets
Sentiment labels: {'LABEL_0': 'negative', 'LABEL_1': 'neutral', 'LABEL_2': 'positive'}


## 2. Dataset Loading API

In [2]:
# Load a sample of the Sentiment140 dataset
# Full dataset has 1.6M tweets - we use 1000 for the API demo
DATASET_PATH = '/app/training.1600000.processed.noemoticon.csv'
df = load_sentiment140(DATASET_PATH, n_samples=1000)
print(f'Loaded {len(df)} tweets')
print(f'Columns: {df.columns.tolist()}')
print(f'Sentiment distribution:')
print(df['sentiment'].value_counts())
print()
print('Sample tweets:')
print(df[['text', 'sentiment']].head(5).to_string())


Loaded 1000 tweets
Columns: ['text', 'label', 'sentiment']
Sentiment distribution:
sentiment
positive    500
negative    500
Name: count, dtype: int64

Sample tweets:
                                                                                                                                            text sentiment
0                                                                                       @indiemoviemaker thanks 4 including me in your shoutout   positive
1                                                                                 No santigold for me tonight. work til 10 and I'm off tomorrow   positive
2  @JonathanRKnight for realz! Safe travels- hope things go much more smoothly this time around.   &amp; that you are having a great day so far!  positive
3                                                                       @thomaskattus you asked about my SF schedule, dahling...maybe next time   positive
4                                                         

## 3. Tweet Preprocessing API

In [3]:
# preprocess_tweet() cleans raw tweets before sending to model
raw_tweets = [
    '@user I love this product! http://example.com',
    'This is terrible, I hate it @user @user2',
    'Just had coffee at Starbucks   ',
]
for tweet in raw_tweets:
    cleaned = preprocess_tweet(tweet)
    print(f'Raw    : {tweet}')
    print(f'Cleaned: {cleaned}')
    print()


Raw    : @user I love this product! http://example.com
Cleaned: @user I love this product!

Raw    : This is terrible, I hate it @user @user2
Cleaned: This is terrible, I hate it @user @user

Raw    : Just had coffee at Starbucks   
Cleaned: Just had coffee at Starbucks



## 4. HuggingFace Sentiment Model API

In [4]:
# Load pre-trained Twitter sentiment model
# cardiffnlp/twitter-roberta-base-sentiment is trained on 58M tweets
from transformers import pipeline

print('Loading sentiment model (this may take a minute on first run)...')
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment',
    tokenizer='cardiffnlp/twitter-roberta-base-sentiment',
    return_all_scores=False,
)
print('Model loaded successfully!')


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentiment model (this may take a minute on first run)...
Model loaded successfully!


/usr/local/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [5]:
# Classify individual tweets
test_tweets = [
    'I absolutely love this! Best day ever!',
    'This is the worst experience I have ever had',
    'Just woke up and having breakfast',
    'So happy about the news today!',
    'I am really disappointed with this service',
]
print('Sentiment classification results:')
print('-' * 60)
for tweet in test_tweets:
    result = sentiment_pipeline(tweet)[0]
    label = SENTIMENT_LABELS.get(result['label'], result['label'])
    score = round(result['score'], 4)
    print(f'Tweet    : {tweet}')
    print(f'Sentiment: {label} (confidence: {score})')
    print()


Sentiment classification results:
------------------------------------------------------------
Tweet    : I absolutely love this! Best day ever!
Sentiment: positive (confidence: 0.9925)

Tweet    : This is the worst experience I have ever had
Sentiment: negative (confidence: 0.9808)

Tweet    : Just woke up and having breakfast
Sentiment: neutral (confidence: 0.7346)

Tweet    : So happy about the news today!
Sentiment: positive (confidence: 0.992)

Tweet    : I am really disappointed with this service
Sentiment: negative (confidence: 0.9815)



## 5. Kafka Producer API

In [6]:
# Connect to Kafka broker
# Note: using kafka:29092 (internal Docker network), not localhost:9092
producer = KafkaProducer(
    bootstrap_servers='kafka:29092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
)
print('Kafka Producer connected successfully!')

# Send a single tweet event
sample_row = df.iloc[0]
event = create_tweet_event(sample_row['text'], sample_row['sentiment'], tweet_id=0)
print(f'Tweet event to send:')
print(json.dumps(event, indent=2))
producer.send(TOPIC_NAME, key='tweet', value=event)
producer.flush()
print('Event sent to Kafka successfully!')


Kafka Producer connected successfully!
Tweet event to send:
{
  "tweet_id": 0,
  "text": "@user thanks 4 including me in your shoutout",
  "true_label": "positive",
  "timestamp": "2026-05-08T23:28:17.715190"
}
Event sent to Kafka successfully!


## 6. Kafka Consumer API

In [7]:
# Read messages from Kafka topic
consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers='kafka:29092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='api-demo-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=3000,
)
messages = []
for msg in consumer:
    messages.append(msg.value)
consumer.close()
print(f'Total messages received: {len(messages)}')
if messages:
    print(f'Sample message:')
    print(json.dumps(messages[0], indent=2))


Total messages received: 0


## 7. Utility Functions API

In [8]:
# compute_sentiment_stats() - computes statistics from classified tweets
sample_results = [
    {'text': 'I love this', 'predicted_sentiment': 'positive', 'true_label': 'positive', 'score': 0.95},
    {'text': 'I hate this', 'predicted_sentiment': 'negative', 'true_label': 'negative', 'score': 0.92},
    {'text': 'Just woke up', 'predicted_sentiment': 'neutral',  'true_label': 'positive', 'score': 0.71},
    {'text': 'Great day!',   'predicted_sentiment': 'positive', 'true_label': 'positive', 'score': 0.88},
    {'text': 'Terrible!',    'predicted_sentiment': 'negative', 'true_label': 'negative', 'score': 0.91},
]
stats = compute_sentiment_stats(sample_results)
print('Sentiment Statistics:')
print(format_sentiment_summary(stats))


Sentiment Statistics:
Total tweets analyzed : 5
Positive              : 2 (40.0%)
Negative              : 2 (40.0%)
Neutral               : 1  (20.0%)
Model accuracy        : 80.0%
